In [0]:
import pandas as pd

df=spark.table("brightcoffee.sales.data").toPandas()
df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,2026-05-30 07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,2026-05-30 07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,2026-05-30 07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,2026-05-30 07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,2026-05-30 07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [0]:
# Parse dates and extract useful columns
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['hour']    = pd.to_datetime(df['transaction_time'], format='%H:%M:%S').dt.hour
df['month']   = df['transaction_date'].dt.month
df['revenue'] = df['unit_price'] * df['transaction_qty']
print(df.head(3))

   transaction_id transaction_date    transaction_time  ...  hour  month revenue
0               1       2023-01-01 2026-05-30 07:06:11  ...     7      1     6.0
1               2       2023-01-01 2026-05-30 07:08:56  ...     7      1     6.2
2               3       2023-01-01 2026-05-30 07:14:04  ...     7      1     9.0

[3 rows x 14 columns]


In [0]:
print('Shape:', df.shape)

Shape: (149116, 14)


In [0]:
print(df.dtypes)

transaction_id               int64
transaction_date    datetime64[ns]
transaction_time    datetime64[ns]
transaction_qty              int64
store_id                     int64
store_location              object
product_id                   int64
unit_price                 float64
product_category            object
product_type                object
product_detail              object
hour                         int32
month                        int32
revenue                    float64
dtype: object


In [0]:
df['revenue'] = df['unit_price'] * df['transaction_qty']
df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,hour,month,revenue
0,1,2023-01-01,2026-05-30 07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg,7,1,6.0
1,2,2023-01-01,2026-05-30 07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,7,1,6.2
2,3,2023-01-01,2026-05-30 07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,7,1,9.0
3,4,2023-01-01,2026-05-30 07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm,7,1,2.0
4,5,2023-01-01,2026-05-30 07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,7,1,6.2


## Task 1 - `if / elif / else`

In [0]:
revenue=8.50

if revenue>10:
    flag='High'
elif revenue>=5:
      flag='Medium'
else:
    flag='Low'
   
print()

print(f'Revenue: R{revenue} → Flag: {flag}')


Revenue: R8.5 → Flag: Medium


In [0]:
revenue=3

if revenue>10:
    flag='High'
elif revenue>=5:
      flag='Medium'
else:
    flag='Low'
   
print(f'Revenue: R{revenue} → Flag: {flag}')

Revenue: R3 → Flag: Low


In [0]:
revenue=10

if revenue>10:
    flag='High'
elif revenue>=5:
      flag='Medium'
else:
    flag='Low'
   
print(f'Revenue: R{revenue} → Flag: {flag}')

Medium



In [0]:
revenue=12

if revenue>10:
    flag='High'
elif revenue>=5:
      flag='Medium'
else:
    flag='Low'


print(f'Revenue: R{revenue} → Flag: {flag}')

Revenue: R12 → Flag: High


## Task 2 - if / elif / else on a DataFrame


In [0]:
def classify_revenue(value):
    if value > 10:
        return 'High'
    elif value >= 5:
        return 'Medium'
    else:
        return 'Low'


df['revenue_flag'] = df['revenue'].apply(classify_revenue)

# How many transactions fall into each category?
print(df['revenue_flag'].value_counts())
print()

# Spot-check: do the labels match the revenue values?
print(df[['revenue', 'revenue_flag']].head(10))

revenue_flag
Low       93463
Medium    52182
High       3471
Name: count, dtype: int64

   revenue revenue_flag
0     6.00       Medium
1     6.20       Medium
2     9.00       Medium
3     2.00          Low
4     6.20       Medium
5     3.00          Low
6     2.00          Low
7     4.00          Low
8     4.25          Low
9     7.00       Medium


## Task 3 - `for` loop

In [0]:

stores = df['store_location'].unique()
print('Stores found:', stores)
print()

for store in stores:
    store_data = df[df['store_location'] == store]
    total = store_data['revenue'].sum()
    print(f'{store} -> Total Revenue: {total:.2f}')

Stores found: ['Lower Manhattan' "Hell's Kitchen" 'Astoria']

Lower Manhattan -> Total Revenue: 230057.25
Hell's Kitchen -> Total Revenue: 236511.17
Astoria -> Total Revenue: 232243.91


##Task 4 - `for` loop with `if` inside

In [0]:
print('Peak hours (more than 3,000 transactions):')
print()

for h in range(23):
    count = len(df[df['hour'] == h])
    if count >3000:
        print(f'  Hour {h:02d}:00  →  {count:,} transactions')

Peak hours (more than 3,000 transactions):

  Hour 06:00  →  4,594 transactions
  Hour 07:00  →  13,428 transactions
  Hour 08:00  →  17,654 transactions
  Hour 09:00  →  17,764 transactions
  Hour 10:00  →  18,545 transactions
  Hour 11:00  →  9,766 transactions
  Hour 12:00  →  8,708 transactions
  Hour 13:00  →  8,714 transactions
  Hour 14:00  →  8,933 transactions
  Hour 15:00  →  8,979 transactions
  Hour 16:00  →  9,093 transactions
  Hour 17:00  →  8,745 transactions
  Hour 18:00  →  7,498 transactions
  Hour 19:00  →  6,092 transactions


## Task 5 - `while` loop

In [0]:
target     = 150_000
month      = 1
cumulative = 0
max_month  = df['month'].max()   # safety — don't go past the data

while cumulative < target and month <= max_month:
    monthly_rev = df[df['month'] == month]['revenue'].sum()
    cumulative += monthly_rev
    print(f'Month {month}: +R{monthly_rev:,.0f}  →  Running total: R{cumulative:,.0f}')
    month += 1

print()
print(f'✓ Target of R{target:,} crossed at month {month - 1}! Final total: R{cumulative:,.0f}')

Month 1: +R81,678  →  Running total: R81,678
Month 2: +R76,145  →  Running total: R157,823

✓ Target of R150,000 crossed at month 2! Final total: R157,823


## Task 6 - Functions

In [0]:
def store_report(store):
    # Filter to just this store
    data  = df[df['store_location'] == store]

    # Calculate stats
    total = data['revenue'].sum()
    count = len(data)
    avg   = total / count

    # Print the report
    print(f'── {store} ──')
    print(f'  Total revenue   : R{total:,.2f}')
    print(f'  Transactions    : {count:,}')
    print(f'  Avg per sale    : R{avg:.2f}')
    print()


# Call the function for each store
store_report("Hell's Kitchen")
store_report('Astoria')
store_report('Lower Manhattan')


── Hell's Kitchen ──
  Total revenue   : R236,511.17
  Transactions    : 50,735
  Avg per sale    : R4.66

── Astoria ──
  Total revenue   : R232,243.91
  Transactions    : 50,599
  Avg per sale    : R4.59

── Lower Manhattan ──
  Total revenue   : R230,057.25
  Transactions    : 47,782
  Avg per sale    : R4.81



## Bonus - Upgrade the Function

In [0]:
def store_report(store):
    # Filter to just this store
    data  = df[df['store_location'] == store]
   
    # Calculate stats
    total = data['revenue'].sum()
    count = len(data)
    avg   = total / count
    top_cat = data.groupby('product_category')['revenue'].sum().idxmax()

    # Print the report
    print(f'── {store} ──')
    print(f'  Total revenue   : R{total:,.2f}')
    print(f'  Transactions    : {count:,}')
    print(f'  Avg per sale    : R{avg:.2f}')
    print(f'  Top product category : {top_cat}')
    print()


# Call the function for each store
store_report("Hell's Kitchen")
store_report('Astoria')
store_report('Lower Manhattan')


── Hell's Kitchen ──
  Total revenue   : R236,511.17
  Transactions    : 50,735
  Avg per sale    : R4.66
  Top product category : Coffee

── Astoria ──
  Total revenue   : R232,243.91
  Transactions    : 50,599
  Avg per sale    : R4.59
  Top product category : Coffee

── Lower Manhattan ──
  Total revenue   : R230,057.25
  Transactions    : 47,782
  Avg per sale    : R4.81
  Top product category : Coffee

